In [1]:
import pandas as pd
import numpy as np
import os
import json
import requests
from pyjstat import pyjstat
from collections import OrderedDict

Eurostat queries based on query builder: https://ec.europa.eu/eurostat/web/json-and-unicode-web-services/getting-started/query-builder

In [2]:
dir_in = "../source_data/Eurostat/"
dir_out = "../parsed_data/"

In [3]:
map_country_ISO = {
    "Belgium" : "BE",
    "Bulgaria" : "BG",
    "Czechia" : "CZ",
    "Denmark" : "DK",
    "Germany (until 1990 former territory of the FRG)" : "DE",
    "Estonia" : "EE",
    "Ireland" : "IE",
    "Greece" : "GR",
    "Spain" : "ES",
    "France" : "FR",
    "Croatia" : "HR",
    "Italy" : "IT",
    "Latvia" : "LV",
    "Lithuania" : "LT",
    "Luxembourg" : "LU",
    "Hungary" : "HU",
    "Netherlands" : "NL",
    "Austria" : "AT",
    "Poland" : "PL",
    "Portugal" : "PT",
    "Romania" : "RO",
    "Slovenia" : "SI",
    "Slovakia" : "SK",
    "Finland" : "FI",
    "Sweden" : "SE",
    "United Kingdom" : "GB",
    "Norway" : "NO"
}

In [4]:
map_tech = {
    'Net electricity generation - Total' : 'Total',
    'Net electricity generation - Hydro' : 'Hydro',
    'Net electricity generation of pure pumped storage hydro plants' : 'Pump',
    'Net electricity generation - Geothermal' : 'Other',
    'Net electricity generation - Nuclear' : 'Nuclear',
    'Net electricity generation from combustion of coal' : 'Coal',
    'Net electricity generation from combustion of oil' : 'Oil',
    'Net electricity generation from combustion of natural gas' : 'Gas',
    'Net electricity generation from combustion of renewables' : 'Biomass',
    'Net electricity generation from combustion of other non-renewable fuels' : 'Other' ,
    'Net electricity generation - Wind' : 'Wind',
    'Net electricity generation from wind onshore' : 'WindOnshore',
    'Net electricity generation from wind offshore' : 'WindOffshore',
    'Net electricity generation from solar radiation' : 'Solar',
    'Net electricity generation from solar photovoltaic installations' : 'Solar',
    'Net electricity generation from solar thermal plants' : 'Solar',
    'Net electricity generation from tide :  wave :  ocean and other non-combustible ,sources' : 'Other',
    'Net electricity generation not elsewhere specified' : 'Other',
    'Used for pumped storage' : 'Pumping'
}

load nrg_105_m which has detailed monthly generation data per technology and country in GWh

In [5]:
indicator = 'nrg_105m'
dataformat = 'json'

params = dict(
    sinceTimePeriod='2017M01',
    precision=1,
    geo = {'AT', 'BE', 'BG', 'CY', 'CZ', 'DE', 'DK', 'EE', 'EL', 'ES', 'FI', 'FR', 'HR', 'HU', 'IE', 'IT', 'LT', 'LU', 'LV', 'MD', 'MK', 'MT', 'NL', 'NO', 'PL', 'PT', 'RO', 'RS', 'SE', 'SI', 'SK', 'TR', 'UA', 'UK', 'AL', 'BA', 'LI', 'IS', 'GE', 'ME', 'XK'},
    unit = 'GWH',
    #INDIC_NRG = {'16_107100B', '16_107101', '16_107101A', '16_107101B', '16_107101C', '16_107102', '16_107103', '16_107104', '16_107104A', '16_107104B', '16_107104C', '16_107104D', '16_107104E', '16_107105', '16_107105A', '16_107105B', '16_107105C', '16_107105C1', '16_107105C2', '16_107105D', '16_107105E'},
    INDIC_NRG = {'16_107100B', '16_107101', '16_107101C', '16_107102', '16_107103', '16_107104A', '16_107104B', '16_107104C', '16_107104D', '16_107104E', '16_107105', '16_107105A', '16_107105B', '16_107105C', '16_107105D', '16_107105E', '17_107302'},
    PRODUCT = '6000'
)

In [6]:
url = 'http://ec.europa.eu/eurostat/wdds/rest/data/v2.1/'+dataformat+'/en/'+indicator+'?'
r = requests.get(url=url, params=params)

In [7]:
df_nrg_105m = pd.DataFrame(pyjstat.from_json_stat(r.json(object_pairs_hook=OrderedDict))[0])
df_nrg_105m = df_nrg_105m.rename(columns={u"time": "time", u"geo": "country",
                                      u"indic_nrg": "tech", u"value": "value",
                                      u"product": "type", "unit":"unit"})
df_nrg_105m['country'] = df_nrg_105m['country'].map(map_country_ISO)
df_nrg_105m['tech'] = df_nrg_105m['tech'].map(map_tech)
df_nrg_105m['MWh'] = df_nrg_105m['value'] * 1000
df_nrg_105m = df_nrg_105m.drop(columns = ['type','unit','value'])
df_nrg_105m = df_nrg_105m.groupby(['tech','country','time']).sum()
df_nrg_105m.head()

INFO:numexpr.utils:NumExpr defaulting to 8 threads.


MWh
tech    country time             
Biomass AT      2017M01  197544.0
                2017M02  185317.0
                2017M03  207824.0
                2017M04  188786.0
                2017M05  197780.0

Some country values are only available on an aggregate basis, so we add them manually, where needed

In [8]:
df_nrg_105m_pivot = df_nrg_105m.reset_index().pivot_table(columns='tech',index=['country','time'],values='MWh')
#Estonia only has onshore wind but is just reported as wind (https://en.wikipedia.org/wiki/Wind_power_in_Estonia):
df_nrg_105m_pivot.WindOnshore[df_nrg_105m_pivot.index.get_level_values('country') == 'EE'] = df_nrg_105m_pivot.Wind[df_nrg_105m_pivot.index.get_level_values('country') == 'EE']
#same for Italy (https://it.wikipedia.org/wiki/Eolico_offshore)
df_nrg_105m_pivot.WindOnshore[df_nrg_105m_pivot.index.get_level_values('country') == 'IT'] = df_nrg_105m_pivot.Wind[df_nrg_105m_pivot.index.get_level_values('country') == 'IT']
df_nrg_105m_pivot.head()

tech              Biomass      Coal        Gas      Hydro  Nuclear       Oil  \
country time                                                                   
AT      2017M01  197544.0  503811.0  2154200.0  2271751.0      0.0  158893.0   
        2017M02  185317.0  393428.0  1493727.0  1970028.0      0.0  100568.0   
        2017M03  207824.0  338195.0   982639.0  2922321.0      0.0   43814.0   
        2017M04  188786.0  247859.0   591899.0  2829567.0      0.0   30752.0   
        2017M05  197780.0  204050.0   219116.0  3786170.0      0.0   26652.0   

tech                Other  Pump   Pumping  Solar      Total      Wind  \
country time                                                            
AT      2017M01  387345.0   0.0  565599.0    0.0  6276637.0  603093.0   
        2017M02  384067.0   0.0  586961.0    0.0  4947672.0  420538.0   
        2017M03  531748.0   0.0  515344.0    0.0  5627746.0  601206.0   
        2017M04  569709.0   0.0  483279.0    0.0  5139040.0  680467.0   
        2017M05  721423.0   0.0  461249.0    0.0  5642045.0  486855.0   

tech             WindOffshore  WindOnshore  
country time                                
AT      2017M01           0.0     603093.0  
        2017M02           0.0     420538.0  
        2017M03           0.0     601206.0  
        2017M04           0.0     680467.0  
        2017M05           0.0     486855.0

In [9]:
df_nrg_105m = pd.DataFrame(df_nrg_105m_pivot.stack()).rename(columns={0:'MWh'})
df_nrg_105m.head()

MWh
country time    tech              
AT      2017M01 Biomass   197544.0
                Coal      503811.0
                Gas      2154200.0
                Hydro    2271751.0
                Nuclear        0.0

In [10]:
df_nrg_105m['year'] = df_nrg_105m.index.get_level_values('time').str[:4]
df_nrg_105m['month'] = df_nrg_105m.index.get_level_values('time').str[5:7]
df_nrg_105m = df_nrg_105m[['year','month','MWh']]
df_nrg_105m.head()

year month        MWh
country time    tech                          
AT      2017M01 Biomass  2017    01   197544.0
                Coal     2017    01   503811.0
                Gas      2017    01  2154200.0
                Hydro    2017    01  2271751.0
                Nuclear  2017    01        0.0

In [11]:
df_nrg_105m_year = df_nrg_105m.groupby(['tech','country','year']).sum().reset_index()
df_nrg_105m_year.head()

,tech,country,year,MWh
0,Biomass,AT,2017,2510588.0
1,Biomass,AT,2018,2518545.0
2,Biomass,AT,2019,2446442.0
3,Biomass,BE,2017,5071496.0
4,Biomass,BE,2018,4800812.0


In [12]:
#export this to CSV
df_nrg_105m.to_csv(dir_out + 'generation_monthly_eurostat.csv',index=True)
df_nrg_105m_year.to_csv(dir_out + 'generation_yearly_eurostat.csv',index=False)